In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import pandas as pd
from tqdm import tqdm
import time

In [2]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def handle_consent_popup(driver):
    """
    Handles the consent popup by clicking the 'Consent' button.
    """
    try:
        # Wait for the "Consent" button to be clickable
        consent_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[@aria-label='Consent']"))
        )
        # Click the "Consent" button
        consent_button.click()
        print("Consent button clicked.")
    except Exception as e:
        print(f"No consent popup found or error occurred: {e}")

In [3]:
def check_checkbox(driver):
    """
    Locates and checks the checkbox with id 'MainContent_autoc' using JavaScript.
    """
    try:
        # Locate the checkbox
        checkbox = driver.find_element(By.ID, "MainContent_autoc")
        
        # Use JavaScript to click the checkbox
        if not checkbox.is_selected():
            driver.execute_script("arguments[0].click();", checkbox)
            print("Checkbox has been checked using JavaScript.")
        else:
            print("Checkbox is already checked.")
    except Exception as e:
        print(f"Error occurred while checking the checkbox: {e}")

In [4]:
def vaani_spell_checker(error_sentence, driver):
    """
    Automates interaction with the Vaani website to correct a Tamil sentence.
    
    Args:
        error_sentence (str): The sentence with spelling errors.
        driver (webdriver): An instance of Selenium WebDriver.
    
    Returns:
        str: The corrected sentence from Vaani.
    """
    try:
        # Open Vaani's website
        driver.get("http://vaani.neechalkaran.com/")
        
        # Wait for the page to load
        time.sleep(10)

        # Handle the consent popup if it appears
        handle_consent_popup(driver)
        check_checkbox(driver)
        
        # Locate the input text box using the 'id' attribute (from the uploaded image)
        input_box = driver.find_element(By.ID, "MainContent_myarea")
        
        # Clear the text box and input the erroneous sentence
        input_box.clear()
        input_box.send_keys(error_sentence)
        
        submit_button = driver.find_element(By.ID, "submit")
        driver.execute_script("arguments[0].click();", submit_button)

        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, "popmenu0"))
        )
        
        # Locate all the <span> elements inside the <li> tags of the <ul>
        span_elements = driver.find_elements(By.XPATH, "//ul[@id='popmenu0']/li/span")
       
        # Use JavaScript to get the inner text of each element
        text_list = [
            driver.execute_script("return arguments[0].textContent;", span).strip()
            for span in span_elements
        ]
        return text_list
    except Exception as e:
        return f"Error: {str(e)}"

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

driver_path = "E:\\4th YEAR\\chromedriver.exe"  # Ensure the path is correct and properly escaped
service = Service(driver_path)  # Create a Service object with the driver path

# Use the service object to initialize the Chrome WebDriver
driver = webdriver.Chrome(service=service)

# Test the vaani_spell_checker function
vaani_spell_checker(u"ஆறம் ஒரு கருவி", driver)

Consent button clicked.
Checkbox has been checked using JavaScript.


['அறம்', 'ஆரம்', 'ஆறாம்', 'ஆறம்', '']

In [18]:
import csv

def write_first_500_rows(input_file, output_file):
    """
    Writes the first 500 rows of a CSV file into another CSV file.

    Args:
        input_file (str): Path to the input CSV file.
        output_file (str): Path to the output CSV file.
    """
    try:
        with open(input_file, mode='r', newline='', encoding='utf-8') as infile, open(output_file, mode='w', newline='', encoding='utf-8') as outfile:

            reader = csv.reader(infile)
            writer = csv.writer(outfile)

            for i, row in enumerate(reader):
                if i == 0 or i <= 500:  # Including the header
                    writer.writerow(row)
                if i == 500:
                    break

        print(f"First 500 rows written to {output_file}")
    except Exception as e:
        print(f"An error occurred: {e}")



In [10]:
import csv

# Input and output file paths
input_file = "E:\\4th YEAR\\ISOLATED ERRORS.txt"  # Replace with your input file path
output_file = "IsolatedErrors.csv"  # Replace with your output file path

# Column headers
headers = ["ErrorSentence", "CorrectSentence", "EditDistance", "ErrorType"]

# Read the tab-separated file and write it to CSV format
with open(input_file, 'r', encoding='utf-8') as tsv_file, open(output_file, 'w', encoding='utf-8', newline='') as csv_file:
    tsv_reader = csv.reader(tsv_file, delimiter='\t')
    csv_writer = csv.writer(csv_file)
    
    # Write headers
    csv_writer.writerow(headers)
    
    # Write rows
    for row in tsv_reader:
        csv_writer.writerow(row)

In [20]:
# Example usage:

write_first_500_rows('IsolatedErrors.csv', 'IsolatedErrors500.csv')


First 500 rows written to IsolatedErrors500.csv


In [47]:
l = ['அறம்', 'ஆரம்', 'ஆறாம்', 'ஆறம்', '']
print(l)
l_emptyRemoved = [i for i in l if i]
print(l_emptyRemoved)
print(l_emptyRemoved[-1])

['அறம்', 'ஆரம்', 'ஆறாம்', 'ஆறம்', '']
['அறம்', 'ஆரம்', 'ஆறாம்', 'ஆறம்']
ஆறம்


In [51]:
sent = u"ஆறம் ஒரு கருவி"
sent.replace("ஆறம்","அறம்")
print(sent)

ஆறம் ஒரு கருவி


In [13]:
def test_spell_checker_with_selenium(corpus_file, output_file, driver):
    """
    Test the Vaani spell checker using Selenium and a corpus file.

    Args:
        corpus_file (str): Path to the CSV file containing the corpus.
        output_file (str): Path to save the evaluation results.
        driver: WebDriver for Chrome.
    """
    import pandas as pd
    from tqdm import tqdm

    # Load the corpus
    corpus = pd.read_csv(corpus_file)
    
    # Add a column for Vaani's corrected output
    corpus["VaaniCorrection"] = ""
    corpus["IsCorrect"] = False
    
    # Evaluate each sentence
    for i, row in tqdm(corpus.iterrows(), total=len(corpus), desc="Evaluating Spell Checker"):
        error_sentence = row["ErrorSentence"]
        correct_sentence = row["CorrectSentence"]
        
        # Initialize corrected_sentence with a default value
        corrected_sentence = error_sentence  # Default: assume no correction

        # Get the corrected sentence from Vaani
        suggestionsList = vaani_spell_checker(error_sentence, driver)
        if suggestionsList:
            suggestionsList_emptyRemoved = [word for word in suggestionsList if word]  # Remove empty suggestions
            if suggestionsList_emptyRemoved:
                errorWord = suggestionsList_emptyRemoved[-1]

                for word in suggestionsList_emptyRemoved[:-1]:
                    suggestedSent = error_sentence.replace(errorWord, word)
                    if suggestedSent == correct_sentence:
                        corrected_sentence = suggestedSent
                        break
        
        # Save the corrected output and comparison result
        corpus.at[i, "VaaniCorrection"] = corrected_sentence
        corpus.at[i, "IsCorrect"] = corrected_sentence == correct_sentence
    
    # Close the WebDriver
    driver.quit()
    
    # Save the results to a new file
    corpus.to_csv(output_file, index=False)
    
    # Calculate and print accuracy
    accuracy = corpus["IsCorrect"].mean()
    print(f"Spell Checker Accuracy: {accuracy:.2%}")

In [14]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

driver_path = "E:\\4th YEAR\\chromedriver.exe"  # Ensure the path is correct and properly escaped
service = Service(driver_path)  # Create a Service object with the driver path

# Use the service object to initialize the Chrome WebDriver
driver = webdriver.Chrome(service=service)

corpus_file = "IsolatedErrors50k.csv"  # Update with the path to your corpus
output_file = "evaluation_results.csv"  # Output file to save results


# Run the evaluation
test_spell_checker_with_selenium(corpus_file, output_file, driver)

Evaluating Spell Checker:   0%|          | 0/50000 [00:00<?, ?it/s]

Consent button clicked.
Checkbox has been checked using JavaScript.


Evaluating Spell Checker:   0%|          | 1/50000 [00:34<481:08:20, 34.64s/it]

No consent popup found or error occurred: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF7912780D5+2992373]
	(No symbol) [0x00007FF790F0BFD0]
	(No symbol) [0x00007FF790DA590A]
	(No symbol) [0x00007FF790DF926E]
	(No symbol) [0x00007FF790DF955C]
	(No symbol) [0x00007FF790E427D7]
	(No symbol) [0x00007FF790E1F3AF]
	(No symbol) [0x00007FF790E3F584]
	(No symbol) [0x00007FF790E1F113]
	(No symbol) [0x00007FF790DEA918]
	(No symbol) [0x00007FF790DEBA81]
	GetHandleVerifier [0x00007FF7912D6A2D+3379789]
	GetHandleVerifier [0x00007FF7912EC32D+3468109]
	GetHandleVerifier [0x00007FF7912E0043+3418211]
	GetHandleVerifier [0x00007FF79106C78B+847787]
	(No symbol) [0x00007FF790F1757F]
	(No symbol) [0x00007FF790F12FC4]
	(No symbol) [0x00007FF790F1315D]
	(No symbol) [0x00007FF790F02979]
	BaseThreadInitThunk [0x00007FFCA9A3259D+29]
	RtlUserThreadStart [0x00007FFCAA94AF38+40]

Checkbox has been checked using JavaScript.


Evaluating Spell Checker:   0%|          | 2/50000 [01:07<465:26:20, 33.51s/it]

...

No consent popup found or error occurred: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF7912780D5+2992373]
	(No symbol) [0x00007FF790F0BFD0]
	(No symbol) [0x00007FF790DA590A]
	(No symbol) [0x00007FF790DF926E]
	(No symbol) [0x00007FF790DF955C]
	(No symbol) [0x00007FF790E427D7]
	(No symbol) [0x00007FF790E1F3AF]
	(No symbol) [0x00007FF790E3F584]
	(No symbol) [0x00007FF790E1F113]
	(No symbol) [0x00007FF790DEA918]
	(No symbol) [0x00007FF790DEBA81]
	GetHandleVerifier [0x00007FF7912D6A2D+3379789]
	GetHandleVerifier [0x00007FF7912EC32D+3468109]
	GetHandleVerifier [0x00007FF7912E0043+3418211]
	GetHandleVerifier [0x00007FF79106C78B+847787]
	(No symbol) [0x00007FF790F1757F]
	(No symbol) [0x00007FF790F12FC4]
	(No symbol) [0x00007FF790F1315D]
	(No symbol) [0x00007FF790F02979]
	BaseThreadInitThunk [0x00007FFCA9A3259D+29]
	RtlUserThreadStart [0x00007FFCAA94AF38+40]

Checkbox has been checked using JavaScript.


Evaluating Spell Checker:   1%|          | 453/50000 [3:52:25<377:29:58, 27.43s/it]

No consent popup found or error occurred: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF7912780D5+2992373]
	(No symbol) [0x00007FF790F0BFD0]
	(No symbol) [0x00007FF790DA590A]
	(No symbol) [0x00007FF790DF926E]
	(No symbol) [0x00007FF790DF955C]
	(No symbol) [0x00007FF790E427D7]
	(No symbol) [0x00007FF790E1F3AF]
	(No symbol) [0x00007FF790E3F584]
	(No symbol) [0x00007FF790E1F113]
	(No symbol) [0x00007FF790DEA918]
	(No symbol) [0x00007FF790DEBA81]
	GetHandleVerifier [0x00007FF7912D6A2D+3379789]
	GetHandleVerifier [0x00007FF7912EC32D+3468109]
	GetHandleVerifier [0x00007FF7912E0043+3418211]
	GetHandleVerifier [0x00007FF79106C78B+847787]
	(No symbol) [0x00007FF790F1757F]
	(No symbol) [0x00007FF790F12FC4]
	(No symbol) [0x00007FF790F1315D]
	(No symbol) [0x00007FF790F02979]
	BaseThreadInitThunk [0x00007FFCA9A3259D+29]
	RtlUserThreadStart [0x00007FFCAA94AF38+40]

Checkbox has been checked using JavaScript.


Evaluating Spell Checker:   1%|          | 453/50000 [3:52:55<424:36:47, 30.85s/it]


KeyboardInterrupt: 

In [21]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

driver_path = "E:\\4th YEAR\\chromedriver.exe"  # Ensure the path is correct and properly escaped
service = Service(driver_path)  # Create a Service object with the driver path

# Use the service object to initialize the Chrome WebDriver
driver = webdriver.Chrome(service=service)

corpus_file = "IsolatedErrors500.csv"  # Update with the path to your corpus
output_file = "evaluation_results.csv"  # Output file to save results


# Run the evaluation
test_spell_checker_with_selenium(corpus_file, output_file, driver)

Evaluating Spell Checker:   0%|          | 0/500 [00:00<?, ?it/s]

Consent button clicked.
Checkbox has been checked using JavaScript.


Evaluating Spell Checker:   0%|          | 1/500 [00:22<3:06:39, 22.44s/it]

No consent popup found or error occurred: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF7912780D5+2992373]
	(No symbol) [0x00007FF790F0BFD0]
	(No symbol) [0x00007FF790DA590A]
	(No symbol) [0x00007FF790DF926E]
	(No symbol) [0x00007FF790DF955C]
	(No symbol) [0x00007FF790E427D7]
	(No symbol) [0x00007FF790E1F3AF]
	(No symbol) [0x00007FF790E3F584]
	(No symbol) [0x00007FF790E1F113]
	(No symbol) [0x00007FF790DEA918]
	(No symbol) [0x00007FF790DEBA81]
	GetHandleVerifier [0x00007FF7912D6A2D+3379789]
	GetHandleVerifier [0x00007FF7912EC32D+3468109]
	GetHandleVerifier [0x00007FF7912E0043+3418211]
	GetHandleVerifier [0x00007FF79106C78B+847787]
	(No symbol) [0x00007FF790F1757F]
	(No symbol) [0x00007FF790F12FC4]
	(No symbol) [0x00007FF790F1315D]
	(No symbol) [0x00007FF790F02979]
	BaseThreadInitThunk [0x00007FFCA9A3259D+29]
	RtlUserThreadStart [0x00007FFCAA94AF38+40]

Checkbox has been checked using JavaScript.


Evaluating Spell Checker:   0%|          | 2/500 [00:55<3:55:52, 28.42s/it]

...

No consent popup found or error occurred: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF7912780D5+2992373]
	(No symbol) [0x00007FF790F0BFD0]
	(No symbol) [0x00007FF790DA590A]
	(No symbol) [0x00007FF790DF926E]
	(No symbol) [0x00007FF790DF955C]
	(No symbol) [0x00007FF790E427D7]
	(No symbol) [0x00007FF790E1F3AF]
	(No symbol) [0x00007FF790E3F584]
	(No symbol) [0x00007FF790E1F113]
	(No symbol) [0x00007FF790DEA918]
	(No symbol) [0x00007FF790DEBA81]
	GetHandleVerifier [0x00007FF7912D6A2D+3379789]
	GetHandleVerifier [0x00007FF7912EC32D+3468109]
	GetHandleVerifier [0x00007FF7912E0043+3418211]
	GetHandleVerifier [0x00007FF79106C78B+847787]
	(No symbol) [0x00007FF790F1757F]
	(No symbol) [0x00007FF790F12FC4]
	(No symbol) [0x00007FF790F1315D]
	(No symbol) [0x00007FF790F02979]
	BaseThreadInitThunk [0x00007FFCA9A3259D+29]
	RtlUserThreadStart [0x00007FFCAA94AF38+40]

Checkbox has been checked using JavaScript.


Evaluating Spell Checker: 100%|█████████▉| 499/500 [4:14:43<00:32, 32.03s/it]

No consent popup found or error occurred: Message: 
Stacktrace:
	GetHandleVerifier [0x00007FF7912780D5+2992373]
	(No symbol) [0x00007FF790F0BFD0]
	(No symbol) [0x00007FF790DA590A]
	(No symbol) [0x00007FF790DF926E]
	(No symbol) [0x00007FF790DF955C]
	(No symbol) [0x00007FF790E427D7]
	(No symbol) [0x00007FF790E1F3AF]
	(No symbol) [0x00007FF790E3F584]
	(No symbol) [0x00007FF790E1F113]
	(No symbol) [0x00007FF790DEA918]
	(No symbol) [0x00007FF790DEBA81]
	GetHandleVerifier [0x00007FF7912D6A2D+3379789]
	GetHandleVerifier [0x00007FF7912EC32D+3468109]
	GetHandleVerifier [0x00007FF7912E0043+3418211]
	GetHandleVerifier [0x00007FF79106C78B+847787]
	(No symbol) [0x00007FF790F1757F]
	(No symbol) [0x00007FF790F12FC4]
	(No symbol) [0x00007FF790F1315D]
	(No symbol) [0x00007FF790F02979]
	BaseThreadInitThunk [0x00007FFCA9A3259D+29]
	RtlUserThreadStart [0x00007FFCAA94AF38+40]

Checkbox has been checked using JavaScript.


Evaluating Spell Checker: 100%|██████████| 500/500 [4:15:14<00:00, 30.63s/it]


Spell Checker Accuracy: 7.20%
